# Assignment 3 – Smart Self-Checkout using Computer Vision
**One-file Google Colab notebook | Python + OpenCV + NumPy**

This notebook is self-contained. It creates a reproducible set of packet-like product images in memory, then demonstrates the techniques required in Tasks 1–10. No external image download is needed. The generated images are **synthetic test fixtures**, not photographs of real canteen products. For final field validation, replace/add them with at least 30 real packet photographs and manually verify labels/outlines as specified in the assignment PDF.

**Run:** Runtime → Run all. The final cells save result tables and example visualizations under `/content/assignment3_outputs/`.

In [ ]:
# 1. Setup
import os, time, math, random, itertools, warnings
from collections import defaultdict
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED)
OUT = "/content/assignment3_outputs"
os.makedirs(OUT, exist_ok=True)
print("OpenCV:", cv2.__version__)
print("Output folder:", OUT)


In [ ]:
# 2. Built-in synthetic product database (12 classes; easy to extend to 500)
# Product artwork uses different base colours, labels, flavour codes and simple graphics.
def make_packet(product_id, variant=0, size=(240, 320)):
    w,h=size
    img=np.full((h,w,3), 245, np.uint8)
    palette=[(30,80,220),(40,170,40),(210,90,20),(180,40,150),
             (20,160,190),(100,70,30),(60,130,210),(160,160,30),
             (90,40,190),(20,110,70),(190,100,80),(70,70,70)]
    c=palette[product_id % len(palette)]
    cv2.rectangle(img,(18,12),(w-18,h-12),c,-1)
    cv2.rectangle(img,(25,20),(w-25,h-20),(245,245,245),3)
    # brand banner and product-specific mark
    cv2.rectangle(img,(35,42),(w-35,92),(250,250,250),-1)
    cv2.putText(img,f"BRAND {product_id//3+1}",(43,75),cv2.FONT_HERSHEY_SIMPLEX,.55,(20,20,20),2)
    cv2.putText(img,["CHIPS","BISCUIT","JUICE"][product_id%3],(38,145),
                cv2.FONT_HERSHEY_DUPLEX,.65,(255,255,255),2)
    cv2.circle(img,(w//2,205),35,(255-c[0],255-c[1],255-c[2]),-1)
    cv2.putText(img,f"V{product_id%3+1}",(w//2-22,211),cv2.FONT_HERSHEY_SIMPLEX,.55,(10,10,10),2)
    cv2.putText(img,f"Rs {10+10*(product_id%3)}",(40,h-35),cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2)
    # subtle texture
    for x in range(32,w-30,24):
        cv2.line(img,(x,105),(x,h-60),(255,255,255),1)
    return img

N_PRODUCTS=30
database={i:make_packet(i) for i in range(N_PRODUCTS)}
def distort(img, angle=0, scale=1.0, brightness=1.0, blur=0, glare=False, occlude=False):
    h,w=img.shape[:2]
    M=cv2.getRotationMatrix2D((w/2,h/2),angle,scale)
    out=cv2.warpAffine(img,M,(w,h),borderValue=(235,235,235))
    out=np.clip(out.astype(np.float32)*brightness,0,255).astype(np.uint8)
    if blur: out=cv2.GaussianBlur(out,(5,5),blur)
    if glare: cv2.ellipse(out,(w//2,h//3),(45,18),-25,0,360,(255,255,255),-1)
    if occlude: cv2.rectangle(out,(w-65,h//2),(w-5,h//2+85),(80,80,80),-1)
    return out

# 4 transformed queries per class = 120 labelled queries
queries=[]
for pid,img in database.items():
    for j,params in enumerate([(12,.95,1.0,0,False,False),(-28,1.15,.85,1,True,False),
                               (35,.8,1.1,0,False,True),(5,1.0,.95,2,True,True)]):
        q=distort(img, *params)
        queries.append((pid,q))
print(f"Database products: {len(database)} | labelled queries: {len(queries)}")


In [ ]:
# 3. Utility functions and preprocessing pipelines (Task 2)
def gray_resize(img, width=160):
    g=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    h=max(1,int(g.shape[0]*width/g.shape[1]))
    return cv2.resize(g,(width,h),interpolation=cv2.INTER_AREA)

def preprocess_A(img):
    return gray_resize(img)

def preprocess_B(img):
    g=gray_resize(img)
    g=cv2.GaussianBlur(g,(5,5),0)
    return cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)).apply(g)

def preprocess_C(img):
    g=preprocess_B(img)
    # suppress saturated glare regions using inpainting
    mask=cv2.threshold(g,248,255,cv2.THRESH_BINARY)[1]
    if cv2.countNonZero(mask)>0:
        g=cv2.inpaint(g,mask,3,cv2.INPAINT_TELEA)
    return g

def sift_detector():
    return cv2.SIFT_create(nfeatures=500) if hasattr(cv2,"SIFT_create") else None
def orb_detector():
    return cv2.ORB_create(nfeatures=500)

def orb_count(img):
    kp=orb_detector().detect(img,None)
    return len(kp)

# Repeatability: nearest keypoint within tolerance after same-view transform approximation
def keypoints_xy(det,img):
    kp=det.detect(img,None)
    return np.array([k.pt for k in kp],dtype=np.float32).reshape(-1,2)

prep_rows=[]
for name,fn in [("A Minimal",preprocess_A),("B Noise+contrast",preprocess_B),("C Full normalization",preprocess_C)]:
    t0=time.perf_counter(); counts=[]
    for pid,img in database.items(): counts.append(orb_count(fn(img)))
    elapsed=(time.perf_counter()-t0)/len(database)*1000
    prep_rows.append([name,round(float(np.mean(counts)),1),round(elapsed,3)])
prep_df=pd.DataFrame(prep_rows,columns=["Pipeline","Mean ORB keypoints","Mean processing ms"])
display(prep_df)


In [ ]:
# 4. Representations and matchers (Tasks 3 and 7)
def pixel_vec(img):
    return cv2.resize(img,(32,32)).astype(np.float32).reshape(-1)
def hsv_hist(img):
    hsv=cv2.cvtColor(img,cv2.COLOR_BGR2HSV)
    hist=cv2.calcHist([hsv],[0,1],None,[16,16],[0,180,0,256])
    cv2.normalize(hist,hist)
    return hist
def sift_desc(img):
    det=sift_detector()
    if det is None: return None
    return det.detectAndCompute(img,None)[1]
def orb_desc(img):
    return orb_detector().detectAndCompute(img,None)[1]

def cosine_sim(a,b):
    a=a.reshape(-1).astype(np.float32); b=b.reshape(-1).astype(np.float32)
    den=np.linalg.norm(a)*np.linalg.norm(b)
    return float(np.dot(a,b)/den) if den else 0.0

def representation_scores(query, rep):
    scores=[]
    for pid,ref in database.items():
        if rep=="pixel":
            a=pixel_vec(query); b=pixel_vec(ref); score=-float(np.linalg.norm(a-b))
        elif rep=="hist":
            score=float(cv2.compareHist(hsv_hist(query),hsv_hist(ref),cv2.HISTCMP_CORREL))
        else:
            qd=orb_desc(query); rd=orb_desc(ref)
            if qd is None or rd is None: score=0
            else:
                matches=cv2.BFMatcher(cv2.NORM_HAMMING,crossCheck=True).match(qd,rd)
                score=len([m for m in matches if m.distance<65])
        scores.append((score,pid))
    return [pid for score,pid in sorted(scores,reverse=True)]

def topk_accuracy(rep, sample=60):
    chosen=queries[:sample]
    hits1=hits5=0
    for truth,q in chosen:
        ranking=representation_scores(q,rep)
        hits1 += ranking[0]==truth
        hits5 += truth in ranking[:5]
    return 100*hits1/len(chosen),100*hits5/len(chosen)

rep_rows=[]
for rep in ["pixel","hist","orb"]:
    a,b=topk_accuracy(rep,60)
    rep_rows.append([rep,round(a,1),round(b,1)])
rep_df=pd.DataFrame(rep_rows,columns=["Representation","Top-1 accuracy %","Top-5 accuracy %"])
display(rep_df)

# SIFT/ORB BF and SIFT+FLANN+RANSAC demonstration on a representative query
def sift_match_count(q, ref, ratio=.75, ransac=True):
    det=sift_detector()
    if det is None: return 0,0
    k1,d1=det.detectAndCompute(q,None); k2,d2=det.detectAndCompute(ref,None)
    if d1 is None or d2 is None or len(d1)<2 or len(d2)<2: return 0,0
    matcher=cv2.FlannBasedMatcher(dict(algorithm=1,trees=5),dict(checks=50))
    pairs=matcher.knnMatch(d1,d2,k=2)
    good=[m for pair in pairs if len(pair)==2 for m,n in [pair] if m.distance < ratio*n.distance]
    inliers=0
    if ransac and len(good)>=4:
        src=np.float32([k1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
        dst=np.float32([k2[m.trainIdx].pt for m in good]).reshape(-1,1,2)
        _,mask=cv2.findHomography(src,dst,cv2.RANSAC,5.0)
        inliers=int(mask.sum()) if mask is not None else 0
    return len(good),inliers

qpid,qimg=queries[5]
print("SIFT+FLANN+RANSAC sample (good matches, inliers):",sift_match_count(qimg,database[qpid]))


In [ ]:
# 5. Edge and corner detection (Tasks 4 and 5)
def edge_maps(img):
    g=preprocess_B(img)
    sobelx=cv2.Sobel(g,cv2.CV_32F,1,0,ksize=3)
    sobely=cv2.Sobel(g,cv2.CV_32F,0,1,ksize=3)
    mag=cv2.convertScaleAbs(cv2.magnitude(sobelx,sobely))
    _,sob=cv2.threshold(mag,70,255,cv2.THRESH_BINARY)
    blur=cv2.GaussianBlur(g,(5,5),1.2)
    log=cv2.convertScaleAbs(cv2.Laplacian(blur,cv2.CV_16S,ksize=3))
    _,log=cv2.threshold(log,35,255,cv2.THRESH_BINARY)
    can=cv2.Canny(g,60,150)
    return sob,log,can

def corner_maps(img):
    g=preprocess_B(img)
    f=np.float32(g)
    harris=cv2.cornerHarris(f,2,3,.04)
    harris=cv2.dilate(harris,None)
    shi=cv2.goodFeaturesToTrack(g,300,.01,5)
    fast=orb_detector().detect(g,None)
    return int(np.sum(harris>.01*harris.max())),0 if shi is None else len(shi),len(fast)

edge_rows=[]; corner_rows=[]
for pid,img in list(database.items())[:10]:
    maps=edge_maps(img)
    for name,m in zip(["Sobel","LoG","Canny"],maps):
        edge_rows.append([name,int(cv2.countNonZero(m))])
    corner_rows.append(["Harris","Shi-Tomasi","FAST",*corner_maps(img)])
edge_df=pd.DataFrame(edge_rows,columns=["Detector","Edge pixels"])
corner_df=pd.DataFrame(corner_rows,columns=["Harris label","Shi-Tomasi label","FAST label","Harris count","Shi-Tomasi count","FAST count"])
display(edge_df.groupby("Detector")["Edge pixels"].mean().round(1).reset_index())
display(corner_df[["Harris count","Shi-Tomasi count","FAST count"]].mean().round(1).to_frame("Mean count"))

# Hough / contour packet outline comparison (Task 6)
def outline_methods(img):
    g=preprocess_B(img); e=cv2.Canny(g,50,140)
    contours,_=cv2.findContours(e,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    contour_img=cv2.cvtColor(img.copy(),cv2.COLOR_BGR2RGB)
    if contours:
        c=max(contours,key=cv2.contourArea)
        peri=cv2.arcLength(c,True); poly=cv2.approxPolyDP(c,.02*peri,True)
        cv2.polylines(contour_img,[poly],True,(255,0,0),2)
    standard=cv2.HoughLines(e,1,np.pi/180,100)
    probabilistic=cv2.HoughLinesP(e,1,np.pi/180,50,minLineLength=35,maxLineGap=10)
    return len(contours),0 if standard is None else len(standard),0 if probabilistic is None else len(probabilistic),contour_img,e
print("Outline method counts:",outline_methods(database[0])[:3])


In [ ]:
# 6. PCA reduction and integrated pipelines (Tasks 8 and 9)
X=np.stack([pixel_vec(database[i]) for i in database]).astype(np.float32)
mean,eigvec=cv2.PCACompute(X,mean=None,maxComponents=64)
projected=cv2.PCAProject(X,mean,eigvec)
explained=np.var(projected,axis=0)
cumvar=np.cumsum(explained)/(np.sum(explained)+1e-9)
print("Components for >=95% variance:",int(np.searchsorted(cumvar,.95)+1))

def pca_rank(q):
    qv=pixel_vec(q).reshape(1,-1)
    qp=cv2.PCAProject(qv,mean,eigvec)
    d=np.linalg.norm(projected-qp,axis=1)
    return np.argsort(d).tolist()

def global_rank(q):
    return pca_rank(q)
def local_rank(q):
    return sorted(database.keys(),key=lambda pid:-representation_scores(q,"orb").index(pid))
def two_stage_rank(q):
    shortlist=global_rank(q)[:10]
    return sorted(shortlist,key=lambda pid:-representation_scores(q,"orb").index(pid))+global_rank(q)[10:]

def evaluate_ranker(ranker, sample=60):
    subset=queries[:sample]; correct=0
    for truth,q in subset: correct += (ranker(q)[0]==truth)
    return 100*correct/len(subset)

pipeline_rows=[]
for name,fn in [("Global PCA",global_rank),("Local ORB",local_rank),("Two-stage",two_stage_rank)]:
    t=time.perf_counter(); acc=evaluate_ranker(fn,30); ms=(time.perf_counter()-t)/30*1000
    pipeline_rows.append([name,round(acc,1),round(ms,2)])
pipeline_df=pd.DataFrame(pipeline_rows,columns=["Pipeline","Top-1 accuracy %","Mean query time ms"])
display(pipeline_df)

# Task 1 solution type classification
print("Low-level: enhance/resize only; mid-level: keypoints/descriptors; high-level: ranked product + confidence + billing decision.")


In [ ]:
# 7. Task 10: tray adaptation – segment items, then classify crops
def make_tray(ids):
    canvas=np.full((520,760,3),235,np.uint8)
    positions=[(30,30),(270,35),(510,30),(100,190),(360,210)]
    for pid,(x,y) in zip(ids,positions):
        p=cv2.resize(database[pid],(180,220))
        canvas[y:y+220,x:x+180]=p
    return canvas

tray=make_tray([0,4,8,12,16])
def segment_tray(img):
    g=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    _,th=cv2.threshold(g,220,255,cv2.THRESH_BINARY_INV)
    th=cv2.morphologyEx(th,cv2.MORPH_CLOSE,np.ones((7,7),np.uint8))
    contours,_=cv2.findContours(th,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    boxes=[]
    for c in contours:
        x,y,w,h=cv2.boundingRect(c)
        if w>70 and h>90: boxes.append((x,y,w,h))
    return sorted(boxes,key=lambda b:(b[1],b[0]))
boxes=segment_tray(tray)
print("Detected tray item regions:",len(boxes),boxes)
plt.figure(figsize=(10,5)); plt.imshow(cv2.cvtColor(tray,cv2.COLOR_BGR2RGB)); plt.title("Built-in multi-item tray fixture"); plt.axis("off"); plt.show()

# Draw sample feature/edge/outline results and save outputs
sample=queries[5][1]
sob,log,can=edge_maps(sample)
fig,axs=plt.subplots(1,3,figsize=(12,4))
for ax,im,title in zip(axs,[sob,log,can],["Sobel","LoG","Canny"]):
    ax.imshow(im,cmap="gray"); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(OUT,"edge_detector_comparison.png"),dpi=150); plt.show()

pd.concat([prep_df.assign(Task="Preprocessing"),rep_df.assign(Task="Representation")],ignore_index=True,sort=False).to_csv(os.path.join(OUT,"summary_results.csv"),index=False)
pipeline_df.to_csv(os.path.join(OUT,"pipeline_results.csv"),index=False)
print("Saved result CSVs and edge visualization to",OUT)


## Interpretation and submission notes
- The notebook's generated packet artwork is a controlled software test dataset. It is useful for verifying that the code runs and for comparing algorithm behaviour, but it does not replace the assignment's required real-world photographs.
- Before final submission, photograph at least 30 actual snack packets; create one clean front-facing reference per product; add labelled query photographs covering rotation, scale, blur, glare, folds and partial finger occlusion.
- Record actual elapsed times on the target counter PC (4 CPU cores, 8 GB RAM, no GPU). Colab timings are not directly comparable to that hardware.
- For a safe billing workflow, require a calibrated confidence threshold and an explicit cashier/student confirmation for low-confidence or out-of-database items. Never automatically charge on an uncertain match.
- This notebook intentionally reports measurements from its own run rather than fabricating human-participant timings or real canteen accuracy.